<a href="https://colab.research.google.com/github/KokYong-Tan/SMU-MDSE/blob/main/week2_computing_colab_Tan_Kok_Yong.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS603 Week 2 Lab — Probability Trees and Bayes' Rule

**Assigned readings (Course Outline, Week 2)**
- **IPDS** 2.2–2.4
- **MPQE** 7.1.1–7.1.3

**Lab checklist**
1. Tree diagram applications (multi-stage experiments, sequential sample spaces, conditionals, LoTP)
2. Bayes' rule applications (prior vs posterior; independence vs conditional independence)

Run all cells in order. No CSV files this week.


## 1. Probability trees and the law of total probability

In [3]:
import math  # math library for exp and factorial in the Poisson likelihood

# --- Stage 1: market regime (partition of the first-stage sample space) ---
P_B = 0.6  # prior / first-stage probability of Boom
P_R = 0.4  # prior / first-stage probability of Recession
assert abs(P_B + P_R - 1.0) < 1e-12  # check that {B, R} is exhaustive (sums to 1)

# --- Stage 2: default given each regime (edge conditionals on the tree) ---
P_D_given_B = 0.05  # P(D | B): default probability in a boom
P_D_given_R = 0.20  # P(D | R): default probability in a recession
P_Dc_given_B = 1.0 - P_D_given_B  # P(D^c | B) by the complement rule
P_Dc_given_R = 1.0 - P_D_given_R  # P(D^c | R) by the complement rule

# --- Leaf probabilities = multiply along each path (multiplication rule) ---
P_BD = P_B * P_D_given_B  # path Boom → Default
P_BDc = P_B * P_Dc_given_B  # path Boom → No default
P_RD = P_R * P_D_given_R  # path Recession → Default
P_RDc = P_R * P_Dc_given_R  # path Recession → No default

leaves = {  # dictionary mapping each sequential outcome to its probability
    "BD": P_BD,  # store Boom-Default leaf
    "BDc": P_BDc,  # store Boom-NoDefault leaf
    "RD": P_RD,  # store Recession-Default leaf
    "RDc": P_RDc,  # store Recession-NoDefault leaf
}

print("Sequential sample space Omega (paths) and leaf probabilities:")  # header
for name, p in leaves.items():  # iterate over each path name and probability
    print(f"  P({name}) = {p:.4f}")  # print one leaf probability to 4 decimals

total = sum(leaves.values())  # sum all leaf probabilities
print(f"Sum of leaf probabilities = {total:.4f} (should be 1)")  # partition check
assert abs(total - 1.0) < 1e-12  # hard check that leaves form a probability partition

# --- Law of total probability for the event Default ---
P_D_lotp = P_D_given_B * P_B + P_D_given_R * P_R  # LoTP: sum_i P(D|regime_i) P(regime_i)
P_D_leaves = P_BD + P_RD  # alternative: sum only the default leaves
print(f"P(D) via LoTP           = {P_D_lotp:.4f}")  # report LoTP result
print(f"P(D) via default leaves = {P_D_leaves:.4f}")  # report leaf-sum result
assert abs(P_D_lotp - P_D_leaves) < 1e-12  # the two methods must agree

P_Dc = 1.0 - P_D_lotp  # complement rule for no-default
print(f"P(D^c) = 1 - P(D)       = {P_Dc:.4f}")  # display complement probability


Sequential sample space Omega (paths) and leaf probabilities:
  P(BD) = 0.0300
  P(BDc) = 0.5700
  P(RD) = 0.0800
  P(RDc) = 0.3200
Sum of leaf probabilities = 1.0000 (should be 1)
P(D) via LoTP           = 0.1100
P(D) via default leaves = 0.1100
P(D^c) = 1 - P(D)       = 0.8900


## 2a. Bayes update: fraud prior → posterior given alert

In [ ]:
P_F = 0.02  # prior probability of fraud
P_Fc = 1.0 - P_F  # prior probability of no fraud (complement)
P_A_given_F = 0.90  # likelihood: alert rate if fraud is true (true positive rate)
P_A_given_Fc = 0.05  # likelihood: alert rate if no fraud (false positive rate)
P_Ac_given_F = 1.0 - P_A_given_F  # P(A^c | F)
P_Ac_given_Fc = 1.0 - P_A_given_Fc  # P(A^c | F^c)

P_A = P_A_given_F * P_F + P_A_given_Fc * P_Fc  # marginal P(A) by LoTP
P_F_given_A = (P_A_given_F * P_F) / P_A  # Bayes: posterior P(F | A)
P_F_given_Ac = (P_Ac_given_F * P_F) / (1.0 - P_A)  # Bayes: posterior P(F | A^c)

print(f"Prior      P(F)       = {P_F:.4f}")  # print the prior
print(f"Evidence   P(A)       = {P_A:.4f}")  # print the marginal probability of an alert
print(f"Posterior  P(F | A)   = {P_F_given_A:.4f}")  # print updated belief given alert
print(f"Posterior  P(F | A^c) = {P_F_given_Ac:.4f}")  # print updated belief given no alert
print("Note: even after an alert, P(F|A) can stay modest when the base rate is low.")  # teaching note


Prior      P(F)       = 0.0200
Evidence   P(A)       = 0.0670
Posterior  P(F | A)   = 0.2687
Posterior  P(F | A^c) = 0.0021
Note: even after an alert, P(F|A) can stay modest when the base rate is low.


## 2b. Bayes update: Poisson claims with a two-point prior on λ

Prior: $P(\Lambda=2)=0.7$, $P(\Lambda=5)=0.3$. Observe $N=4$ from $\mathrm{Poisson}(\Lambda)$.


In [ ]:
prior = {2: 0.7, 5: 0.3}  # two-point prior: P(Λ=2)=0.7, P(Λ=5)=0.3
k_obs = 4  # observed claim count N = 4

def poisson_pmf(k: int, lam: float) -> float:
    return math.exp(-lam) * (lam**k) / math.factorial(k)  # e^{-λ} λ^k / k!

lik = {lam: poisson_pmf(k_obs, lam) for lam in prior}  # likelihood P(N=k | Λ=λ) for each atom
evidence = sum(lik[lam] * prior[lam] for lam in prior)  # P(N=k) = sum_λ lik(λ) prior(λ)
posterior = {lam: lik[lam] * prior[lam] / evidence for lam in prior}  # Bayes renormalisation

print(f"Observed N = {k_obs}")  # report the data
for lam in prior:  # loop over the two rate values
    print(  # print prior, likelihood, and posterior side by side
        f"  λ={lam}: prior={prior[lam]:.3f}, "
        f"lik={lik[lam]:.5f}, posterior={posterior[lam]:.3f}"
    )
print(f"Evidence P(N={k_obs}) = {evidence:.5f}")  # print the marginal likelihood
assert abs(sum(posterior.values()) - 1.0) < 1e-12  # posterior must be a probability distribution


Observed N = 4
  λ=2: prior=0.700, lik=0.09022, posterior=0.545
  λ=5: prior=0.300, lik=0.17547, posterior=0.455
Evidence P(N=4) = 0.11580


## 2c. Independence vs conditional independence (two loans)

Common-factor model: regime $C\in\{B,R\}$; given $C$, defaults $D_1,D_2$ are independent.


In [12]:
# Common-factor model: regime C, then conditionally independent defaults
P_B = 0.6  # P(Boom)
P_R = 0.4  # P(Recession)
p_D_B = 0.05  # P(D_i | B) for each loan i = 1,2
p_D_R = 0.25  # P(D_i | R) for each loan i = 1,2

# Conditional independence given Boom: joint = product of margins
P_both_given_B = p_D_B * p_D_B  # P(D1 ∩ D2 | B) under D1 ⊥ D2 | B
P_both_given_R = p_D_R * p_D_R  # P(D1 ∩ D2 | R) under D1 ⊥ D2 | R
print(f"P(D1 ∩ D2 | B) = {P_both_given_B:.4f} (= {p_D_B}*{p_D_B})")  # show conditional joint
print(f"P(D1 ∩ D2 | R) = {P_both_given_R:.4f} (= {p_D_R}*{p_D_R})")  # show conditional joint

# Unconditional margins via LoTP
P_D1 = p_D_B * P_B + p_D_R * P_R  # P(D1) = sum_c P(D1|c) P(c)
P_D2 = P_D1  # loans are exchangeable here, so P(D2)=P(D1)
P_both = P_both_given_B * P_B + P_both_given_R * P_R  # P(D1 ∩ D2) by LoTP over regimes
prod_margins = P_D1 * P_D2  # product P(D1)P(D2) that independence would require

print(f"P(D1) = P(D2) = {P_D1:.4f}")  # print unconditional default probability
print(f"P(D1 ∩ D2)           = {P_both:.4f}")  # unconditional joint
print(f"P(D1) P(D2)          = {prod_margins:.4f}")  # independence benchmark
print(f"Independent?         = {abs(P_both - prod_margins) < 1e-12}")  # False if dependence
print("Conclusion: defaults are conditionally independent given the regime,")  # narrative line 1
print("but unconditionally dependent because they share the common factor C.")  # narrative line 2


P(D1 ∩ D2 | B) = 0.0025 (= 0.05*0.05)
P(D1 ∩ D2 | R) = 0.0625 (= 0.25*0.25)
P(D1) = P(D2) = 0.1300
P(D1 ∩ D2)           = 0.0265
P(D1) P(D2)          = 0.0169
Independent?         = False
Conclusion: defaults are conditionally independent given the regime,
but unconditionally dependent because they share the common factor C.


## Exercises

**E1.** Add a third regime $S$ (stagflation) with $P(S)=0.1$; rescale $P(B)$, $P(R)$ so probabilities still sum to 1. Choose $P(D\mid S)$ and recompute $P(D)$ via LoTP.


In [22]:
# --- Stage 1: market regime with a third regime (Stagflation) ---
P_S = 0.1  # probability of Stagflation

# Rescale P_B and P_R so probabilities still sum to 1 with P_S
P_B = 0.55
P_R = 0.35

assert abs(P_B + P_R + P_S - 1.0) < 1e-12 # check that {B, R, S} is exhaustive (sums to 1)

# --- Stage 2: default given each regime (edge conditionals on the tree) ---
P_D_given_B = 0.05  # P(D | B): default probability in a boom (keeping original value)
P_D_given_R = 0.20  # P(D | R): default probability in a recession (keeping original value)
P_D_given_S = 0.10  # P(D | S): chosen default probability in stagflation

print(f"New P(Boom) = {P_B:.4f}")
print(f"New P(Recession) = {P_R:.4f}")
print(f"P(Stagflation) = {P_S:.4f}")
print(f"P(D | S) = {P_D_given_S:.4f}")

# --- Recompute Law of total probability for the event Default ---
P_D_lotp_new = (P_D_given_B * P_B) + (P_D_given_R * P_R) + (P_D_given_S * P_S)

print(f"\nP(D) via LoTP (with Stagflation) = {P_D_lotp_new:.4f}")

New P(Boom) = 0.5500
New P(Recession) = 0.3500
P(Stagflation) = 0.1000
P(D | S) = 0.1000

P(D) via LoTP (with Stagflation) = 0.1075


## Exercises

**E2.** Using the fraud numbers above, compute  P(F∣Ac)  by hand and compare to the cell output; interpret the change from the prior.


In [20]:
P_F = 0.02  # Prior probability of fraud
P_Fc = 1.0 - P_F  # Prior probability of no fraud
P_A_given_F = 0.90  # P(A | F): Alert rate if fraud (True Positive Rate)
P_A_given_Fc = 0.05  # P(A | Fc): Alert rate if no fraud (False Positive Rate)

# P(Ac | F) = 1 - P(A | F)
P_Ac_given_F = 1.0 - P_A_given_F  # P(No Alert | Fraud) : No alert when fraud (False Negative)

# P(Ac | Fc) = 1 - P(A | Fc)
P_Ac_given_Fc = 1.0 - P_A_given_Fc  # P(No Alert | No Fraud) : No alert when no fraud (True Negative)

# Calculate P(Ac) using the Law of Total Probability:
# P(Ac) = P(Ac | F) * P(F) + P(Ac | Fc) * P(Fc)
P_Ac = (P_Ac_given_F * P_F) + (P_Ac_given_Fc * P_Fc)

# Calculate P(F | Ac) using Bayes' Rule:
# P(F | Ac) = [P(Ac | F) * P(F)] / P(Ac)
P_F_given_Ac = (P_Ac_given_F * P_F) / P_Ac

print(f"Manual calculation of P(F | Ac) = {P_F_given_Ac:.4f}") # print updated belief given no alert
print(f"Prior P(F)                      = {P_F:.4f}")

print(f"\nInterpretation:")
print(f"The prior probability of fraud P(F) was 0.02.")
print(f"After observing no alert, the posterior probability P(F | Ac) becomes 0.0021.")
print("This indicates that the absence of an alert significantly reduces the hypothesis that fraud is present.")


Manual calculation of P(F | Ac) = 0.0021
Prior P(F)                      = 0.0200

Interpretation:
The prior probability of fraud P(F) was 0.02.
After observing no alert, the posterior probability P(F | Ac) becomes 0.0021.
This indicates that the absence of an alert significantly reduces the hypothesis that fraud is present.


## Exercises

**E3.** In the two-loan model, print  P(D1∩D2∣B)  and  P(D1∩D2∣R) , then compare  P(D1∩D2)  with  P(D1)P(D2)  to confirm unconditional dependence.

In [21]:
# Run Lab Exercise 2C prior to running this

print(f"P(D1∩D2∣B) = {P_both_given_B:.4f}")
print(f"P(D1∩D2∣R) = {P_both_given_R:.4f}")
print(f"P(D1∩D2) = {P_both:.4f}")
print(f"P(D1)P(D2) = {prod_margins:.4f}")
print(f"\nAre P(D1∩D2) & P(D1)P(D2) equal? {P_both == prod_margins}")
print(f"Conclusion: Since P(D1∩D2) is not equal to P(D1)P(D2), the defaults are unconditionally dependent.")

P(D1∩D2∣B) = 0.0025
P(D1∩D2∣R) = 0.0625
P(D1∩D2) = 0.0265
P(D1)P(D2) = 0.0169

Are P(D1∩D2) & P(D1)P(D2) equal? False
Conclusion: Since P(D1∩D2) is not equal to P(D1)P(D2), the defaults are unconditionally dependent.
